# NORT Inference — Apply Trained Classifier to Novel Videos

This notebook applies a trained SVM classifier to all NOVEL-phase NORT videos to produce per-frame exploration predictions.

**Inputs:**
- `svm_exploratory_classifier.pkl` — trained RBF-SVM
- `feature_scaler.pkl` — fitted StandardScaler from training
- NOVEL-phase `.mp4` videos + their DLC `.csv` files
- Per-video `_bbox_assignments.pkl` files (bounding box positions for each frame)

**Pipeline overview:**
1. Load the trained model and scaler
2. For each NOVEL video, extract the same geometric features used during training
3. Predict exploratory / non-exploratory for each frame
4. If exploratory, assign the exploration to the closest object by nose-to-centroid distance
5. Save per-video JSON prediction files and labeled MP4s with overlaid annotations

**Output:** Per-video `_predictions.json` files aggregated into `nort_results.csv` for use in `Correlate-MoSeq-And-Traditional-Behavior.ipynb`

In [ ]:
import json
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
import os
import shutil
from tqdm import tqdm
import joblib

from nort import inference

# 1. Load Trained Model

Load the RBF-SVM classifier and fitted `StandardScaler` saved by `Novel-Object-Recognition-Test.ipynb`. The scaler must be the same one used during training — applying it here ensures feature distributions match what the model was trained on.

In [ ]:
model_dir = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/nort_model"
model_file = os.path.join(model_dir, "svm_exploratory_classifier.pkl")
scaler_file = os.path.join(model_dir, "feature_scaler.pkl")

svm_optimized = joblib.load(model_file)
scaler = joblib.load(scaler_file)

print("Model and scaler loaded!")
print(f"Model: {type(svm_optimized)}")
print(f"Scaler: {type(scaler)}")

# 2. Inference Pipeline

**`inference.predict_with_object_assignment`** — the core inference function. For each frame:
1. Load the per-frame bbox assignment and DLC coordinates (matching by video stem, stripping any `DLC...` run suffix from the filename)
2. Compute the 16-dimensional whole-frame feature vector (both objects concatenated)
3. Predict exploratory / non-exploratory using the SVM
4. If exploratory, compute nose-to-centroid distance for each object and assign to the closest one

Returns a dict with per-frame predictions, probabilities, assigned objects, and distances.

In [ ]:
dlc_folder = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/Revision-NORT-Viewable"
bbox_assignments_dir = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/bbox_assignments"
results_dir = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/results_nort"

all_videos = [f for f in os.listdir(dlc_folder) if f.endswith('.mp4')]
novel_videos = [v for v in all_videos if 'NOVEL' in v.upper()]

# 3. Select NOVEL Videos

Filter the video directory to NOVEL-phase recordings only (filenames containing `'NOVEL'`). Only the NOVEL phase is scored — the FAMILIAR phase is used for habituation and is not relevant to the discrimination index.

# 4. Debug: Bbox File Matching

Some NOVEL video filenames include a DLC-run suffix (e.g. `...DLC_Resnet101_NORTDec17shuffle2_..._labeled`) appended to the base video name, while bbox files were saved using only the base stem before the `DLC` suffix. Check coverage first, since `inference.predict_with_object_assignment` already strips that suffix when looking up the bbox file.

In [ ]:
bbox_files = sorted([f for f in os.listdir(bbox_assignments_dir) if f.endswith('.pkl')])

print(f"Total bbox files: {len(bbox_files)}")
print(f"\nFirst 10 bbox files:")
for i, f in enumerate(bbox_files[:10]):
    print(f"  {i+1}. {f}")

novel_video_stems = [Path(v).stem for v in novel_videos]

missing_bbox = []
has_bbox = []

for video_stem in novel_video_stems:
    bbox_stem = video_stem.split('DLC')[0] if 'DLC' in video_stem else video_stem
    expected_bbox = f"{bbox_stem}_bbox_assignments.pkl"
    if expected_bbox in bbox_files:
        has_bbox.append(video_stem)
    else:
        missing_bbox.append(video_stem)

print(f"\n{'='*80}")
print(f"NOVEL videos WITH bbox: {len(has_bbox)}")
print(f"NOVEL videos MISSING bbox: {len(missing_bbox)}")

if len(missing_bbox) > 0:
    print(f"\nFirst 5 missing:")
    for i, v in enumerate(missing_bbox[:5]):
        print(f"  {i+1}. {v}")

novel_videos_with_bbox = [v for v, stem in zip(novel_videos, novel_video_stems) if stem in has_bbox]

print(f"\nNOVEL videos with bbox files: {len(novel_videos_with_bbox)} / {len(novel_videos)}")
print(f"Processing only these {len(novel_videos_with_bbox)} videos...")

# 5. Run Inference and Save Results

Writes, per video:
- `<video_stem>_predictions.json` — frame-level predictions, probabilities, assigned object, distances
- `<video_stem>_labeled.mp4` — annotated video with overlaid exploratory/non-exploratory labels

In [ ]:
print("Processing all NOVEL videos...")
inference.process_and_save_novel_videos(
    novel_videos_with_bbox,
    svm_optimized,
    scaler,
    dlc_folder,
    bbox_assignments_dir,
    results_dir,
)
print("\nAll NOVEL videos processed!")